In [1]:
import os
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 2048


url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
path = "/media/zman/extrahd/reu20024project/qastuff/output_file.jsonl"

dataset = load_dataset("json", data_files = {"train" : path}, split = "train")



# Load the dataset
dataset = load_dataset("json", data_files= {"train": path}, split="train")

# Randomly split into 80% training and 20% validation
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

# Extract the train and validation sets
train_dataset = split_dataset['train']
validation_dataset = split_dataset['test']

# Verify the sizes of the split datasets
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")



# 2. Load Llama3 model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3 Before training
def generate_text(text, model):
    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    outputs = model.generate(**inputs, max_new_tokens=20)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("Before training\n")
#generate_text("<human>: What is PROTECT?.\n<bot>: ")

# 4. Do model patching and add fast LoRA weights and training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # Rank stabilized LoRA
    loftq_config = None, # LoftQ
)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset=validation_dataset, 
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 50,
        num_train_epochs = 3,  # Specify the number of epochs here
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        evaluation_strategy="steps",  # Enable evaluation during training
        eval_steps=50,  # Evaluate every 50 steps
        output_dir = "outputs",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
    ),
)
trainer.train()



/home/zman/anaconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Generating train split: 2060 examples [00:00, 223366.58 examples/s]


Training dataset size: 1648
Validation dataset size: 412
==((====))==  Unsloth 2024.10.5: Fast Llama patching. Transformers = 4.45.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.475 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.0. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Unsloth: We fixed a gradient accumulation bug, but it seems like you don't have the latest transformers version!
Please update transformers, TRL and unsloth via:
`pip install --upgrade --no-cache-dir unsloth git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git`


Before training



Unsloth 2024.10.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
/home/zman/anaconda3/envs/unsloth_env/lib/python3.11/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/zman/anaconda3/envs/unsloth_env/lib/python3.11/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Map: 100%|██████████| 412/412 [00:00<00:00, 22587.75 examples/s]
max_steps is given, it will override any value given in num_train_epochs
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,648 | Num Epochs = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 400
 "-____-"     Number 

**** Unsloth: Please use our fixed gradient_accumulation_steps by updating transformers, TRL and Unsloth!
`pip install --upgrade --no-cache-dir unsloth git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git`


Step,Training Loss,Validation Loss
50,1.720100,1.690279
100,1.637900,1.539979
150,1.284100,1.509145
200,1.949200,1.486288
250,1.402100,1.478948
300,1.675500,1.472075
350,1.454000,1.467824
400,1.720200,1.465882


TrainOutput(global_step=400, training_loss=1.5826309204101563, metrics={'train_runtime': 617.4385, 'train_samples_per_second': 5.183, 'train_steps_per_second': 0.648, 'total_flos': 8869592122146816.0, 'train_loss': 1.5826309204101563, 'epoch': 1.941747572815534})

In [2]:


# 2. Load Llama3 model
original_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
# Call for_inference to initialize the model for inference
original_model = FastLanguageModel.for_inference(original_model)
model = FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2024.10.5: Fast Llama patching. Transformers = 4.45.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.475 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.0. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


In [3]:
# 3 Before training
def generate_text(text, model):
    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    outputs = model.generate(**inputs, max_new_tokens=500)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [4]:
# 5. After training

print("\n ######## \Before training\n")
generate_text("<human>:What contaminants are found in Puerto Rico in PROTECT?\n<bot>: ", original_model)
print("\n ######## \nAfter training\n")
generate_text("<human>:What kind of data does PROTECT collect from participants? \n<bot>: ", model)


 ######## \Before training

<human>:What contaminants are found in Puerto Rico in PROTECT?
<bot>: 1. Lead, 2. Mercury, 3. Arsenic, 4. Chloroform, 5. Cadmium, 6. Manganese, 7. Benzo(a)pyrene, 8. Dichloromethane, 9. Nickel, 10. Bisphenol-A, 11. Acrylamide, 12. Bromoform, 13. 1,4-dioxane, 14. Tetrachloroethylene, 15. 1,1-dichloroethylene, 16. Trichloroethylene, 17. 1,2-dichloropropane, 18. 1,2-dichloroethane, 19. 1,1,1-trichloroethane, 20. Chloroform, 21. Hexachlorobenzene, 22. Heptachlor, 23. Dieldrin, 24. Alpha-Hexachlorocyclohexane, 25. Beta-Hexachlorocyclohexane, 26. Hexachlorobenzene, 27. Polychlorinated biphenyls, 28. Mirex, 29. Polychlorinated biphenyls, 30. Polychlorinated biphenyls, 31. Polychlorinated biphenyls, 32. Polychlorinated biphenyls, 33. Polychlorinated biphenyls, 34. Polychlorinated biphenyls, 35. Polychlorinated biphenyls, 36. Polychlorinated biphenyls, 37. Polychlorinated biphenyls, 38. Polychlorinated biphenyls, 39. Polychlorinated biphenyls, 40. Polychlorinated bi

In [5]:
print("\n ######## Before training\n")
generate_text("<human>: Tell me about the PROTECT center in Puerto Rico\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>: tell me about the PROTECT center in Puerto Rico\n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:What contaminants are found in Puerto Rico in PROTECT?\n<bot>: ", original_model)
print("\n ######## \nAfter training\n")
generate_text("<human>:What contaminants are found in Puerto Rico in PROTECT? \n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:What is the PROTECT Center?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:What is the PROTECT Center? \n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:How many participants are there in PROTECT?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:How many participants are there in PROTECT? \n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:When did PROTECT start?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:When did PROTECT start? \n<bot>: ", model)


 ######## Before training

<human>: Tell me about the PROTECT center in Puerto Rico
<bot>:  What is the PROTECT center in Puerto Rico?
<human>:  It is a center in Puerto Rico that is dedicated to helping children and families.
<bot>:  What is the PROTECT center in Puerto Rico?
<human>:  It is a center in Puerto Rico that is dedicated to helping children and families.
<bot>:  What is the PROTECT center in Puerto Rico?
<human>:  It is a center in Puerto Rico that is dedicated to helping children and families.
<bot>:  What is the PROTECT center in Puerto Rico?
<human>:  It is a center in Puerto Rico that is dedicated to helping children and families.
<bot>:  What is the PROTECT center in Puerto Rico?
<human>:  It is a center in Puerto Rico that is dedicated to helping children and families.
<bot>:  What is the PROTECT center in Puerto Rico?
<human>:  It is a center in Puerto Rico that is dedicated to helping children and families.
<bot>:  What is the PROTECT center in Puerto Rico?
<human

In [6]:

print("\n ######## After training\n")
generate_text("<human>: How many participants in protect in 2015?  \n<bot>: ", model)


 ######## After training

<human>: How many participants in protect in 2015?  
<bot>: 3,000 pregnant women in 2015. 1,000 were from Puerto Rico, 1,000 from the US Virgin Islands, and 1,000 from the continental United States. 1,000 were from Puerto Rico, 1,000 from the US Virgin Islands, and 1,000 from the continental United States. 1,000 were from Puerto Rico, 1,000 from the US Virgin Islands, and 1,000 from the continental United States.
